

databricks free: https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/2460720313805618?o=7474657872577658#command/8032192150355725


aulas do databricks

https://community.databricks.com/t5/learning-events/virtual-learning-festival-10-october-31-october-2025/ev-p/127652

https://community.databricks.com/t5/learning-events/advanced-learning-festival-15-june-06-july-2026/ev-p/156932



Developing Applications with Apache Spark™


https://customer-academy.databricks.com/learn/courses/3962/developing-applications-with-apache-spark?hash=d2245ade317149fcb8f6d57ad991d9aa7f055f9f&generated_by=409740



# Working with Complex Data Types in Apache Spark

# 4 - Working with Complex Data Types in Spark

This demonstration shows how to effectively work with nested data structures in Spark, including structs, arrays, and maps, using real e-commerce data examples.

## Objectives

* Work with arrays, maps, and JSON data within Spark using the DataFrame API.

# 4 - Working with Complex Data Types in Spark

This demonstration shows how to effectively work with nested data structures in Spark, including structs, arrays, and maps, using real e-commerce data examples.

## Objectives

* Convert JSON string data to Spark SQL native complex types
* Understand and manipulate complex data types (Struct, Array, Map)
* Process nested JSON-like data structures
* Use the pivot and explode functions to reshape datasets as required

---

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that Serverless is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use Serverless.



2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:
   * In the drop-down, select **More**.
   * In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.



**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.
2. Find the triangle icon to the right of your compute cluster name and click it.
3. Wait a few minutes for the cluster to start.
4. Once the cluster is running, complete the steps above to select your cluster.

---

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It will set your default catalog to `dbacademy` and the schema to your specific schema name shown below using the `USE` statements.

Also, it will create a temp table for you named `raw_user_data`.

```sql
USE CATALOG dbacademy;
USE SCHEMA dbacademy_<your_unique_schema_name>;

#### NOTE: The DA object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course.   

In [0]:
#%run ./Includes/Classroom-Setup-04

```
%sql
SELECT * FROM raw_user_data

```

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW raw_user_data_teste AS
SELECT * FROM VALUES
  ('100', 'Alice Souza', 'true',  '2024-01-15', '["sports", "music", "travel"]', '[{"item": "laptop", "price": 1200.0}]'),
  ('101', 'Bruno Lima', 'true',  '2024-03-02', '["movies", "tech"]', '[{"item": "mouse", "price": 45.5}, {"item": "keyboard", "price": 89.9}]'),
  ('102', 'Carla Dias', 'false', '2024-05-20', '["cooking"]',                    '[]')
AS (user_id, name, active, signup_date, interests, recent_purchases);

SELECT * FROM raw_user_data_teste

user_id,name,active,signup_date,interests,recent_purchases
100,Alice Souza,true,2024-01-15,"[""sports"", ""music"", ""travel""]","[{""item"": ""laptop"", ""price"": 1200.0}]"
101,Bruno Lima,true,2024-03-02,"[""movies"", ""tech""]","[{""item"": ""mouse"", ""price"": 45.5}, {""item"": ""keyboard"", ""price"": 89.9}]"
102,Carla Dias,false,2024-05-20,"[""cooking""]",[]


In [0]:
%sql
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
workspace,default


## B. Convert from JSON Strings to StructTypes

Given raw data which includes nested JSON strings (arrays and/or objects), we will convert this data to native `StructTypes` in the DataFrame API.

### Why Convert JSON Strings to StructTypes?

JSON strings in Spark DataFrames come with several inefficiencies:

1. **Parsing Overhead:** Every time you query JSON strings, Spark needs to parse them, adding computational overhead.
2. **Memory Inefficiency:** JSON strings store field names repeatedly for every row, wasting memory.
3. **No Type Safety:** JSON strings don't enforce data types, leading to potential errors.
4. **Poor Query Performance:** Spark can't optimize queries on JSON string content as effectively.
5. **Limited Predicate Pushdown:** Filter operations can't leverage columnar storage optimizations.

### Steps to Convert JSON Strings to StructTypes

1. **Infer Schema:** Determine the structure of the JSON data (the `schema_of_json` function can be used for this).
2. **Apply Schema:** Use `from_json()` to convert strings to structured data.
3. **Validate:** Ensure all data is correctly parsed and types are appropriate.
4. **Optimize:** Once converted, optimize storage/processing if needed.

## Benefits of StructTypes

1. **Columnar Storage:** Efficient storage with Parquet/Delta.
2. **Type Safety:** Schema enforcement prevents data errors.
3. **Query Optimization:** Spark can optimize queries better with typed data.
4. **Predicate Pushdown:** Filters can be pushed down to storage layer.
5. **Better Performance:** Faster queries and reduced memory usage.


In [0]:
# python
from pyspark.sql.functions import *

# Load some data which includes JSON strings
raw_user_data_df = spark.read.table("raw_user_data_teste")

In [0]:
# Inspect the data
raw_user_data_df.printSchema()

root
 |-- user_id: string (nullable = false)
 |-- name: string (nullable = false)
 |-- active: string (nullable = false)
 |-- signup_date: string (nullable = false)
 |-- interests: string (nullable = false)
 |-- recent_purchases: string (nullable = false)



In [0]:
# Displaying the dataframe
display(raw_user_data_df)

user_id,name,active,signup_date,interests,recent_purchases
100,Alice Souza,true,2024-01-15,"[""sports"", ""music"", ""travel""]","[{""item"": ""laptop"", ""price"": 1200.0}]"
101,Bruno Lima,true,2024-03-02,"[""movies"", ""tech""]","[{""item"": ""mouse"", ""price"": 45.5}, {""item"": ""keyboard"", ""price"": 89.9}]"
102,Carla Dias,false,2024-05-20,"[""cooking""]",[]


1. View the data above and find columns `interests` and `recent_purchases`.

2. `interests` is an array column of String elements and `recent_purchases` is a column containing objects.

`

In [0]:
# ``python
# Interests is an array of strings, using predefined schema
interests_schema = ArrayType(StringType()) # array lista

In [0]:
# Let's get the schema for the "recent_purchases" and cast all of the columns from the raw data set into a confirmed structure

# Take a sample of one value of the "recent_purchases" column, bring this back to the Driver
recent_purchases_json = raw_user_data_df.select("recent_purchases").limit(1).collect()[0][0]
print("Raw JSON string:", recent_purchases_json)

Raw JSON string: [{"item": "laptop", "price": 1200.0}]


**Use the `schema_of_json` Function to generate a Schema based upon a sample row of data**

In many cases, especically with multiple nested complex structures, it is easiest to generate a schema based upon a sample of JSON data, we can do this using the `schema_of_json` Function.

From the above dataset, we can see that `interests` is an array column of string elements, `recent_purchases` is an array column which contains objects.



In [0]:
#```python
# Get the schema for the recent_purchases JSON
recent_purchases_schema = schema_of_json(lit(recent_purchases_json)) # dicionário

In [0]:
recent_purchases_schema

Column<'schema_of_json([{"item": "laptop", "price": 1200.0}])'>

In [0]:
# Parse columns with the correct schemas
parsed_users_df = raw_user_data_df.select(
    col("user_id").cast("integer"),
    col("name"),
    col("active").cast("boolean"),
    col("signup_date").cast("date"),
    from_json(col("interests"), interests_schema).alias("interests"),
    from_json(col("recent_purchases"), recent_purchases_schema).alias("recent_purchases")
)

# Examine the schema
parsed_users_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- name: string (nullable = false)
 |-- active: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- interests: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- recent_purchases: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item: string (nullable = true)
 |    |    |-- price: double (nullable = true)



In [0]:
display(parsed_users_df)

user_id,name,active,signup_date,interests,recent_purchases
100,Alice Souza,true,2024-01-15,"List(sports, music, travel)","List(List(laptop, 1200.0))"
101,Bruno Lima,true,2024-03-02,"List(movies, tech)","List(List(mouse, 45.5), List(keyboard, 89.9))"
102,Carla Dias,false,2024-05-20,List(cooking),List()


In [0]:
# Now look at the data again, you will notice order has been changed for recent_purchases
display(parsed_users_df)

user_id,name,active,signup_date,interests,recent_purchases
100,Alice Souza,true,2024-01-15,"List(sports, music, travel)","List(List(laptop, 1200.0))"
101,Bruno Lima,true,2024-03-02,"List(movies, tech)","List(List(mouse, 45.5), List(keyboard, 89.9))"
102,Carla Dias,false,2024-05-20,List(cooking),List()


# C. Working with Arrays

Let's explore different ways to access and manipulate arrays.

In [0]:
# Use the array_size method to see the lengths of the array columns in our
display(
    parsed_users_df.select(
        "user_id",
        array_size("interests").alias("number_of_interests"),
        array_size("recent_purchases").alias("number_of_recent_purchases")
    )
)

user_id,number_of_interests,number_of_recent_purchases
100,3,1
101,2,2
102,1,0


# 1. The Explode Method

The explode method is used to unnest array elements into records.

In [0]:

# Let's start by simplifying the data by selecting only the columns we need
user_101s_interests_df = parsed_users_df.select("user_id", "interests").filter(parsed_users_df.user_id == 101)
display(user_101s_interests_df)

user_id,interests
101,"List(movies, tech)"


In [0]:
# Let's demonstrate explode, note how there are three rows associated with "user_id" 101 (one for each "interests" array element)
display(
    user_101s_interests_df.select(
        "user_id",
        explode("interests").alias("interest")
    )
)

user_id,interest
101,movies
101,tech


## 2. The `collect_set` and `collect_list` Methods

The `collect_set` and `collect_list` methods are aggregate functions (which typically operate on grouped data) to create arrays from column values.

`collect_list` may include duplicate values, while `collect_set` removes duplicate array elements should they exist.


In [0]:
# Let's start by creating a new DataFrame with the "interests" column exploded
exploded_df = parsed_users_df.select("user_id", explode("interests").alias("interest"))
display(exploded_df)

user_id,interest
100,sports
100,music
100,travel
101,movies
101,tech
102,cooking


In [0]:
# Let's use `collect_list` to collect all the "interest" values into a list for each "user_id"
user_interests_df = exploded_df.groupBy("user_id").agg(collect_list("interest").alias("interests"))

In [0]:
display(user_interests_df)

user_id,interests
100,"List(sports, music, travel)"
101,"List(movies, tech)"
102,List(cooking)


## D. Referencing Struct Fields

Let's explore how to access fields within a struct (an object with a predefined schema).



In [0]:
display(parsed_users_df)

user_id,name,active,signup_date,interests,recent_purchases
100,Alice Souza,true,2024-01-15,"List(sports, music, travel)","List(List(laptop, 1200.0))"
101,Bruno Lima,true,2024-03-02,"List(movies, tech)","List(List(mouse, 45.5), List(keyboard, 89.9))"
102,Carla Dias,false,2024-05-20,List(cooking),List()


In [0]:
#```python
# First let's explode the "recent_purchases" column
exploded_purchases_df = parsed_users_df.select("user_id", explode("recent_purchases").alias("purchase"))
display(exploded_purchases_df)

user_id,purchase
100,"List(laptop, 1200.0)"
101,"List(mouse, 45.5)"
101,"List(keyboard, 89.9)"


In [0]:
# Use the dot notation to access struct fields
#recent_purchases_df = exploded_purchases_df.select(
#    "user_id",
#    col("purchase.date").alias("purchase_date"),
#    col("purchase.product_id").alias("product_id"),
#    col("purchase.name").alias("product_name"),
#    col("purchase.price").alias("purchase_price")
#)


# Use the dot notation to access struct fields
recent_purchases_df = exploded_purchases_df.select(
    "user_id",
    col("purchase.item").alias("purchase_item"),
    col("purchase.price").alias("product_price")
)

display(recent_purchases_df)

user_id,purchase_item,product_price
100,laptop,1200.0
101,mouse,45.5
101,keyboard,89.9


In [0]:
# We can also use the getField() method to reference columns inside of structs, here's an example....

#field_access_df = exploded_purchases_df.select(
#    "user_id",
#    col("purchase").getField("date").alias("purchase_date"),
#    col("purchase").getField("product_id").alias("product_id"),
#    col("purchase").getField("name").alias("product_name"),
#    col("purchase").getField("price").alias("price")
#)

# We can also use the getField() method to reference columns inside of structs, here's an example....

field_access_df = exploded_purchases_df.select(
    "user_id",
    col("purchase").getField("item").alias("purchase_item"),
    col("purchase").getField("price").alias("price")
)

display(field_access_df)

user_id,purchase_item,price
100,laptop,1200.0
101,mouse,45.5
101,keyboard,89.9


## E. Using the `pivot` Method

The `pivot` method in Spark allows you to transform row data into columnar format, creating a cross-tabulation. This is particularly useful for feature engineering when analyzing categorical data or when you need to reshape your data for reporting or visualization.



In [0]:
# Let's pivot the purchase data to show the count of each product purchased by each user
#pivot_df = (recent_purchases_df
#    .groupBy("user_id")
#    .pivot("product_item")
#    .agg(count("product_price").alias("quantity_purchased"))
#)



In [0]:
#```python
# Let's pivot the purchase data to show the count of each product purchased by each user
pivot_df = (recent_purchases_df
    .groupBy("user_id")
    .pivot("purchase_item")
    .agg(count("product_price").alias("quantity_purchased"))
)

# Display the result
display(pivot_df)

user_id,keyboard,laptop,mouse
100,null,1,null
101,1,null,1


In [0]:
# Replace null values with zeros for better readability
pivot_df_no_nulls = (recent_purchases_df
    .groupBy("user_id")
    .pivot("purchase_item")
    .agg(count("product_price").alias("quantity_purchased"))
    .fillna(0)
)

# Display the result
display(pivot_df_no_nulls)

user_id,keyboard,laptop,mouse
100,0,1,0
101,1,0,1
